# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the [FAIR^2 mass spectrometry](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described using a [Croissant schema](https://mlcommons.org/croissant/) and accessible from the following URL:

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Title: {metadata.name}\nDescription: {metadata.description}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs (`@id`).

Let's enumerate the entities that make up the dataset, focusing on the record sets, fields, and columns by their `@id`. This step is essential for data exploration and sets up later extraction and analysis by ensuring all references use the proper and unique `@id` values.


In [ ]:
# List all available record sets
print("Record Sets and their fields/columns (@id):\n")
record_sets = dataset.metadata.record_sets

all_record_set_ids = []
for rs in record_sets:
    print(f"RecordSet: {rs.name} (@id: {rs.id})")
    all_record_set_ids.append(rs.id)
    if hasattr(rs, 'fields') and rs.fields:
        print("  Fields:")
        for field in rs.fields:
            print(f"    - {field.name} (@id: {field.id}, type: {field.data_type})")
    if hasattr(rs, 'columns') and rs.columns:
        print("  Columns:")
        for col in rs.columns:
            print(f"    - {col.name} (@id: {col.id}, type: {col.data_type})")
    print()

## 3. Data Extraction
Load data from specific record sets into pandas DataFrames for analysis.

**Note:** All references use the `@id` of the record sets and fields.

In [ ]:
# You can adjust record_sets_to_extract based on the overview above. For demonstration, load all record sets.
record_sets_to_extract = all_record_set_ids  # List of all record set @ids found previously

dataframes = {}
for record_set_id in record_sets_to_extract:
    print(f"Loading records from RecordSet @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Fields: {dataframes[record_set_id].columns.tolist()}")
        display(dataframes[record_set_id].head())
    else:
        print("No records found for this record set.\n")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering, normalization, transformation, grouping, and preparation for further analysis.

**Choose a specific record set and field for numeric analysis.**

*For demonstration, we select the first record set that contains at least one numeric field.*

In [ ]:
import numpy as np

# Helper: find first numeric field in loaded DataFrames
numeric_field_id = None
record_set_id = None

# Searching for the first numeric field in the loaded DataFrames
for rs in dataset.metadata.record_sets:
    if rs.id in dataframes and not dataframes[rs.id].empty:
        # Find numeric field among columns or fields
        for field in (getattr(rs, 'fields', []) or []) + (getattr(rs, 'columns', []) or []):
            if field.data_type in ['Float', 'Number', 'Integer', 'schema:Float', 'schema:Number', 'schema:Integer']:
                if field.id in dataframes[rs.id].columns:
                    numeric_field_id = field.id
                    record_set_id = rs.id
                    break
    if numeric_field_id:
        break

if numeric_field_id is None or record_set_id is None:
    print("No numeric field found in loaded dataframes! EDA cannot proceed.")
else:
    print(f"Using RecordSet: {record_set_id}")
    print(f"Numeric field: {numeric_field_id}")
    df = dataframes[record_set_id].copy()
    # Coerce to numeric, drop NAs
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    df = df.dropna(subset=[numeric_field_id])

    # Filtering
    threshold = df[numeric_field_id].mean()  # Use mean for demonstration
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"].copy()])

    # Attempt grouping by a categorical field if present
    group_field_id = None
    # Choose first string/category field that's not the numeric field
    for field in (getattr(rs, 'fields', []) or []) + (getattr(rs, 'columns', []) or []):
        if field.id != numeric_field_id and field.id in df.columns:
            if field.data_type in ['Text', 'schema:Text'] or df[field.id].dtype == object:
                group_field_id = field.id
                break
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"Grouped data by {group_field_id} (mean {numeric_field_id}):")
        display(grouped_df.head())

## 5. Visualization
Visualize the distribution of the selected numeric field and, if possible, how it varies by a group field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and record_set_id:
    plt.figure(figsize=(7, 4))
    sns.histplot(df[numeric_field_id], bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field_id:
        # Bar plot of mean values by group
        plt.figure(figsize=(8, 4))
        group_means = df.groupby(group_field_id)[numeric_field_id].mean().sort_values(ascending=False).head(10)
        sns.barplot(y=group_means.index, x=group_means.values, orient='h')
        plt.xlabel(f"Mean {numeric_field_id}")
        plt.ylabel(group_field_id)
        plt.title(f"Top 10 groups by mean {numeric_field_id}")
        plt.show()

## 6. Conclusion
This notebook demonstrated how to:

- Load FAIR^2 mass spectrometry analysis data using the Croissant schema and the `mlcroissant` library.
- Reference all entities via their `@id`, ensuring fully reproducible and auditable data processing.
- Explore all available record sets and their fields.
- Extract all records from the dataset and encapsulate them in pandas DataFrames.
- Perform basic exploratory data analysis, including filtering, normalization, and grouping.
- Visualize numeric distributions and group means.

Further analysis can be tailored to specific biological, proteomic, or domain research questions by selecting and processing the relevant record sets, fields, and columns—always using Croissant `@id`s.
